In [2]:
import cv2
import easyocr
import re

image = cv2.imread("input3.jpg")

# STEP 1: preprocessing  by grayscaling and 
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
blurred = cv2.GaussianBlur(gray, (5, 5), 0)

#storing the processed image in a new file
cv2.imwrite("output3.jpg", blurred)

#OCR
reader = easyocr.Reader(['en'])
results = reader.readtext(blurred)

def clean_text(text):
    text = re.sub(r"\d+\.\d+", "", text)   # remove confidence-like numbers
    text = re.sub(r"\d+", "", text)        # remove prices/numbers
    return text.strip()

#array to store all food items
foodItems = []

#words we don't need in our list
JUNK_WORDS = [
    "FOOD", "MENU", "RESTAURANT", "INC", "INC.", "CO", "LTD","DELICIOUS FOOD",
    
    "APPETIZER", "APPETIZERS", "STARTERS", "SOUPS", "SALADS",
    "MAIN COURSES", "MAINS", "ENTREES",
    "DESSERTS", "DRINKS", "BEVERAGES",
    "SIDES",

    "SPECIALS", "CHEF SPECIALS", "TODAY SPECIAL",
    "COMBO", "PLATTERS",

]

for (_, text, conf) in results:
    if conf > 0.5:  # filtering bad OCR results
        cleaned = clean_text(text)

        if len(cleaned) > 2 and cleaned.upper() not in JUNK_WORDS:
            foodItems.append(cleaned)

print(", ".join(foodItems))

/opt/anaconda3/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Mushroom Burger, Crispy Fried Chicken, Fish & Chips, Spaghetti & Meatballs, Hotdog Sandwich, SALAD & SOUP, Macaroni Soup, Tex Mex Chili, French Fries, Calamari, Beef Taco, D RINKS, Purified Water, Sparkling Water, Soda In A Bottle, Orange Juice, Fresh Lemonade


In [3]:
import pandas as pd

# Function to check for both allergens and food restrictions
def check_food_safety(csv_file, food_items, user_allergens, user_restrictions):
    # 1. Load the dataset
    df = pd.read_csv(csv_file)
    
    # 2. Combine the user's interests and filter the database
    # This ensures we only look for groups the user cares about
    interesting_groups = user_allergens + user_restrictions
    relevant_df = df[df['group_name'].isin(interesting_groups)]
    
    print(f"--- Scanning for Allergens: {', '.join(user_allergens)} ---")
    print(f"--- Scanning for Restrictions: {', '.join(user_restrictions)} ---\n")
    
    found_any = False
    
    # 3. Go through each food item provided in the input list
    for item in food_items:
        item_lower = item.lower()
        matches = []
        
        # 4. Search the ingredient column for keywords found in the food item name
        for _, row in relevant_df.iterrows():
            ingredient_keyword = str(row['ingredient']).lower()
            
            # Check if the ingredient keyword exists within the food item name
            if ingredient_keyword in item_lower:
                matches.append({
                    'ingredient': row['ingredient'],
                    'category': row['group_name'],
                    'group_type': row['group_type'],
                    'recommendation': row.get('ingredient_type', 'N/A')
                })
        
        # 5. Format and print the results
        if matches:
            # We use a set of tuples to remove any identical duplicate matches
            unique_matches = sorted(list({ 
                (m['ingredient'], m['category'], m['group_type'], m['recommendation']) 
                for m in matches 
            }))
            
            for ing, cat, g_type, rec in unique_matches:
                print(f"Matched Food: '{item}'")
                print(f"  -> Ingredient Found: '{ing}'")
                print(f"  -> Category: {cat} ({g_type.replace('_', ' ').title()})")
                
                # If it's a restriction, show if the user should avoid it
                if g_type == 'food_restriction':
                    print(f"  -> Recommendation: {rec.replace('_', ' ').title()}")
                
                print("-" * 30)
            found_any = True

    if not found_any:
        print("No matching allergens or restrictions found for the provided items.")

# --- INPUT SECTION ---

# 1. List of food items to check

# 2. List of specific allergens to watch out for
my_allergens = [
    "milk", "egg", "peanut", "tree_nut", "soy", "wheat", 
    "gluten", "fish", "shellfish", "sesame", "mustard", 
    "celery", "corn", "coconut", "garlic", "onion", "legume"
]

# 3. List of food restrictions to watch out for
my_restrictions = ['vegan', 'vegetarian', 'halal', 'kosher']

# Run the function
check_food_safety('allergen_database.csv', foodItems, foodItems, my_restrictions)

--- Scanning for Allergens: Mushroom Burger, Crispy Fried Chicken, Fish & Chips, Spaghetti & Meatballs, Hotdog Sandwich, SALAD & SOUP, Macaroni Soup, Tex Mex Chili, French Fries, Calamari, Beef Taco, D RINKS, Purified Water, Sparkling Water, Soda In A Bottle, Orange Juice, Fresh Lemonade ---
--- Scanning for Restrictions: vegan, vegetarian, halal, kosher ---

Matched Food: 'Crispy Fried Chicken'
  -> Ingredient Found: 'chicken'
  -> Category: halal (Food Restriction)
  -> Recommendation: Allowed With Condition
------------------------------
Matched Food: 'Crispy Fried Chicken'
  -> Ingredient Found: 'chicken'
  -> Category: kosher (Food Restriction)
  -> Recommendation: Allowed With Condition
------------------------------
Matched Food: 'Crispy Fried Chicken'
  -> Ingredient Found: 'chicken'
  -> Category: vegan (Food Restriction)
  -> Recommendation: Avoid
------------------------------
Matched Food: 'Crispy Fried Chicken'
  -> Ingredient Found: 'chicken'
  -> Category: vegetarian (Fo